In [ ]:
"""
    pip install transformers torch torchvision pillow requests
"""
 


from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import torch
import sys
import os
import requests
from io import BytesIO

# ---------------------------------------------------------------------------
# Load Pretrained Model
# ---------------------------------------------------------------------------

MODEL_NAME = "nlpconnect/vit-gpt2-image-captioning"

print("Loading model (first run will download ~1GB weights)...")
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded successfully on {device}.\n")

# ---------------------------------------------------------------------------
# Caption Generation
# ---------------------------------------------------------------------------

def generate_caption(image: Image.Image, max_length: int = 64, num_beams: int = 4) -> str:
    """Generate a caption for a PIL Image."""
    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        output_ids = model.generate(
            pixel_values,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True
        )

    caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return caption.strip()

# ---------------------------------------------------------------------------
# Image Loading Helpers
# ---------------------------------------------------------------------------

def load_image_from_path(path: str) -> Image.Image:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    return Image.open(path)

def load_image_from_url(url: str) -> Image.Image:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content))

# ---------------------------------------------------------------------------
# Interactive CLI
# ---------------------------------------------------------------------------

def main():
    print("=" * 55)
    print("       CodSoft Image Captioning AI")
    print("=" * 55)
    print("Commands:")
    print("  path  → caption a local image file")
    print("  url   → caption an image from a URL")
    print("  quit  → exit\n")

    while True:
        choice = input("Enter mode (path / url / quit): ").strip().lower()

        if choice == "quit":
            print("Goodbye! 👋")
            break

        elif choice == "path":
            path = input("Enter image file path: ").strip()
            try:
                image = load_image_from_path(path)
                print("Generating caption...")
                caption = generate_caption(image)
                print(f"\n📸 Caption: {caption}\n")
            except FileNotFoundError as e:
                print(f"Error: {e}")
            except Exception as e:
                print(f"Unexpected error: {e}")

        elif choice == "url":
            url = input("Enter image URL: ").strip()
            try:
                image = load_image_from_url(url)
                print("Generating caption...")
                caption = generate_caption(image)
                print(f"\n📸 Caption: {caption}\n")
            except Exception as e:
                print(f"Error loading image from URL: {e}")

        else:
            print("Invalid choice. Type 'path', 'url', or 'quit'.")

if __name__ == "__main__":
    main()